#**Brain Tumor MRI Classification using CNN**

##Step 1: Import Required Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

##Step 2: Define Paths and Parameters

In [ ]:
data_dir = "data"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25
NUM_CLASSES = 4

##Step 3: Data Augmentation and Preprocessing

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

class_names = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

##Step 4: Create Data Loaders

In [ ]:
train_generator = train_datagen.flow_from_directory(
    "/data/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_generator = val_test_datagen.flow_from_directory(
    "/data/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)


##STEP 5: Load Pretrained Model (Transfer Learning)

In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

##STEP 6: Freeze Base Layers

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    layer.trainable = True

##STEP 7: Build Custom Classification Head

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

##STEP 8: Compile the Model

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

##STEP 9: Define Callbacks

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

checkpoint = ModelCheckpoint(
    filepath="best_brain_tumor_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=False
)

##STEP 10: Train the Model

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, lr_scheduler, checkpoint]
)

##STEP 11: Save the Final Model

In [ ]:
model.save("best_brain_tumor_model.keras")

##STEP 12: Load Saved Model

In [ ]:
from tensorflow.keras.models import load_model

loaded_model = load_model("best_brain_tumor_model.keras")

##STEP 13: Evaluate on Test Data

In [ ]:
test_generator = val_test_datagen.flow_from_directory(
    "/data/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_loss, test_acc = loaded_model.evaluate(test_generator)
print("Test Accuracy:", test_acc*100, "%")

##STEP 14: Detailed Evaluation

In [ ]:
y_pred = loaded_model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)

y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred_classes, target_names=class_names))

##STEP 15: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(6,5))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks(range(NUM_CLASSES), class_names, rotation=45)
plt.yticks(range(NUM_CLASSES), class_names)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

##STEP 16: Predict a Single MRI Image

In [ ]:
def predict_single_image(img_path, model):
    # Load image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)

    # Display image
    plt.imshow(img_array.astype("uint8"))
    plt.axis("off")
    plt.title("Input MRI Image")
    plt.show()

    # Preprocess
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    # Predict
    predictions = model.predict(img_array)
    predicted_class = np.argmax(predictions)
    confidence = np.max(predictions)

    print(f"Predicted Class : {class_names[predicted_class]}")
    print(f"Confidence      : {confidence * 100:.2f}%")